# ViT-B/16 PatchCore x224 - Main Benchmark

Self-contained training and evaluation notebook for the ViT-B/16 PatchCore run on WM-811K at native 224x224 resolution.

**Flags (set at top of next cell):**
- `RETRAIN = False` - load pre-saved artifacts; set `True` to re-run training from scratch (requires GPU + `LSWMD.pkl`)
- `REGENERATE_UMAP = False` - display pre-saved UMAP PNGs; set `True` to recompute from saved embeddings
- `RUN_HOLDOUT = True` - include secondary holdout evaluation (70 k normals / 3.5 k defects)


## Submission Context

- Dataset notebook: `data/dataset/x224/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x224/benchmark_50k_5pct/data_config.toml`
- Artifact root: `experiments/anomaly_detection/patchcore/vit_b16/x224/main/artifacts/patchcore_vit_b16_5pct/main_5pct`
- Holdout root: `experiments/anomaly_detection/patchcore/vit_b16/x224/main/artifacts/patchcore_vit_b16_5pct/holdout70k_3p5k`
- Backbone: `vit_base_patch16_224.augreg_in21k_ft_in1k` (timm), block-6 hook, 196 patch tokens
- Mode: artifact-first review with checked-in checkpoint (`RETRAIN = False`)


In [ ]:
from __future__ import annotations
import copy
import json
import os
import pickle
import random
import sys
import time
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.core.indexes as core_indexes
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
torch.backends.cudnn.benchmark = True
RETRAIN = False
REGENERATE_UMAP = False
RUN_HOLDOUT = False
LABEL_NORMAL = 'none'
LABEL_DEFECT = 'pattern'

def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            return candidate.resolve()
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate.resolve()
    return Path.cwd().resolve()
REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
CONFIG: dict[str, Any] = {'run': {'variant_name': 'vit_b16_one_layer_patchcore_x224', 'raw_pickle': 'data/raw/LSWMD.pkl', 'output_dir': 'experiments/anomaly_detection/patchcore/vit_b16/x224/main/artifacts/patchcore_vit_b16_5pct/main_5pct', 'seed': 42}, 'data': {'image_size': 224, 'train_normal_count': 40000, 'val_normal_count': 5000, 'test_normal_count': 5000, 'test_defect_count': 250, 'holdout_normal_count': 70000, 'holdout_defect_count': 3500, 'batch_size': 128, 'num_workers': 0}, 'model': {'backbone': 'vit_base_patch16_224.augreg_in21k_ft_in1k', 'vit_feature_block': 6, 'patch_embed_dim': 128, 'memory_bank_max_patches': 400000, 'score_chunk': 512, 'patchcore_nn_k': 3, 'topk_patch_ratio': 0.1}, 'scoring': {'threshold_quantile': 0.95}, 'umap': {'metric': 'cosine', 'pca_components': 32, 'n_neighbors': 15, 'min_dist': 0.1, 'knn_k': 15, 'max_train_reference': 5000, 'max_val_normal': 5000, 'max_test_normal': 5000, 'max_test_anomaly': 250}}
OUTPUT_DIR = (REPO_ROOT / CONFIG['run']['output_dir']).resolve()
HOLDOUT_DIR = OUTPUT_DIR.parent / 'holdout70k_3p5k'
EVAL_DIR = OUTPUT_DIR / 'results' / 'evaluation'
PLOTS_DIR = OUTPUT_DIR / 'plots'
CHKPT_DIR = OUTPUT_DIR / 'checkpoints'
UMAP_DIR = OUTPUT_DIR / 'results' / 'umap'
H_EVAL_DIR = HOLDOUT_DIR / 'results' / 'evaluation'
H_PLOTS_DIR = HOLDOUT_DIR / 'plots'
H_UMAP_DIR = HOLDOUT_DIR / 'results' / 'umap'
for _d in [OUTPUT_DIR, EVAL_DIR, PLOTS_DIR, CHKPT_DIR, UMAP_DIR]:
    _d.mkdir(parents=True, exist_ok=True)
print(f'Repo root  : {REPO_ROOT}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'Holdout dir: {HOLDOUT_DIR}')
print(f'RETRAIN={RETRAIN}  REGENERATE_UMAP={REGENERATE_UMAP}  RUN_HOLDOUT={RUN_HOLDOUT}')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def read_legacy_pickle(path: Path) -> pd.DataFrame:
    sys.modules['pandas.indexes'] = core_indexes
    with path.open('rb') as f:
        return pickle.load(f, encoding='latin1')

def unwrap_legacy_value(value: Any) -> str:
    if value is None:
        return ''
    if hasattr(value, 'size') and getattr(value, 'size') == 0:
        return ''
    if hasattr(value, 'tolist'):
        value = value.tolist()
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    return str(value).strip()

def normalize_map(wafer_map: np.ndarray, image_size: int) -> np.ndarray:
    import torch.nn.functional as F
    arr = np.asarray(wafer_map, dtype=np.float32) / 2.0
    t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return F.interpolate(t, size=(image_size, image_size), mode='nearest').squeeze().numpy()

def build_split_from_raw(raw_path: Path, image_size: int, train_n: int, val_n: int, test_normal_n: int, test_defect_n: int, seed: int) -> dict[str, np.ndarray]:
    df = read_legacy_pickle(raw_path).copy()
    df['failureTypeText'] = df['failureType'].map(unwrap_legacy_value)

    def infer_label(row: pd.Series) -> str | None:
        ft = unwrap_legacy_value(row.get('failureType', '')).lower()
        if ft == 'none':
            return LABEL_NORMAL
        if ft:
            return LABEL_DEFECT
        return None
    df['label'] = df.apply(infer_label, axis=1)
    df = df[df['label'].notna()].reset_index(drop=True)
    normal_df = df[df['label'] == LABEL_NORMAL].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    defect_df = df[df['label'] == LABEL_DEFECT].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    train_df = normal_df.iloc[:train_n]
    val_df = normal_df.iloc[train_n:train_n + val_n]
    test_normal_df = normal_df.iloc[train_n + val_n:train_n + val_n + test_normal_n]
    test_defect_df = defect_df.iloc[:test_defect_n]
    test_df = pd.concat([test_normal_df, test_defect_df], ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    def to_arrays(frame: pd.DataFrame, tag: str):
        arrays = np.stack([normalize_map(np.asarray(row['waferMap']), image_size) for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f'prep_{tag}', leave=False)])
        labels = (frame['label'].values == LABEL_DEFECT).astype(np.int64)
        dtypes = frame['failureTypeText'].fillna('unlabeled').replace('', 'unlabeled').to_numpy()
        return (arrays, labels, dtypes)
    train_x, train_y, train_dt = to_arrays(train_df, 'train')
    val_x, val_y, val_dt = to_arrays(val_df, 'val')
    test_x, test_y, test_dt = to_arrays(test_df, 'test')
    print(f'Split | train={len(train_y)} | val={len(val_y)} | test_normal={int((test_y == 0).sum())} | test_defect={int((test_y == 1).sum())}')
    return {'train_x': train_x, 'train_y': train_y, 'train_defect_types': train_dt, 'val_x': val_x, 'val_y': val_y, 'val_defect_types': val_dt, 'test_x': test_x, 'test_y': test_y, 'test_defect_types': test_dt}

class ViTArrayDataset(Dataset):

    def __init__(self, arrays: np.ndarray, labels: np.ndarray, image_size: int=224) -> None:
        self.arrays = arrays.astype(np.float32)
        self.labels = labels.astype(np.int64)
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        x = torch.from_numpy(self.arrays[idx])
        states = torch.clamp(torch.round(x * 2.0), 0, 2).to(dtype=torch.long)
        one_hot = F.one_hot(states, num_classes=3).permute(2, 0, 1).float()
        if one_hot.shape[-1] != self.image_size:
            one_hot = F.interpolate(one_hot.unsqueeze(0), size=(self.image_size, self.image_size), mode='nearest').squeeze(0)
        return (one_hot, torch.tensor(self.labels[idx], dtype=torch.long))

class ViTB16PatchCoreModel(nn.Module):
    """ViT-B/16 PatchCore: block-6 patch token hook, 768->128 cosine-distance memory bank."""
    PATCHES_PER_IMAGE = 196
    VIT_EMBED_DIM = 768

    def __init__(self, *, feature_block: int=6, patch_embed_dim: int=128, nn_k: int=3, topk_patch_ratio: float=0.1, score_chunk: int=512) -> None:
        super().__init__()
        import timm
        self.feature_block = int(feature_block)
        self.patch_embed_dim = int(patch_embed_dim)
        self.nn_k = int(nn_k)
        self.topk_patch_ratio = float(topk_patch_ratio)
        self.score_chunk = int(score_chunk)
        backbone = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0)
        self.patch_embed = backbone.patch_embed
        self.cls_token = backbone.cls_token
        self.pos_embed = backbone.pos_embed
        self.pos_drop = backbone.pos_drop
        self.blocks = backbone.blocks[:self.feature_block + 1]
        self.norm = backbone.norm
        self.proj = nn.Linear(self.VIT_EMBED_DIM, self.patch_embed_dim, bias=False)
        self.register_buffer('memory_bank', torch.empty(0, self.patch_embed_dim))
        for p in list(self.patch_embed.parameters()) + [self.cls_token, self.pos_embed] + list(self.blocks.parameters()) + list(self.norm.parameters()):
            if isinstance(p, nn.Parameter):
                p.requires_grad_(False)
            else:
                p.requires_grad_(False)
        self.proj.weight.requires_grad_(False)

    def _forward_patch_tokens(self, x: torch.Tensor) -> torch.Tensor:
        """Return patch tokens at block-6 output: shape [B, 196, 768]."""
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x[:, 1:, :]

    def patch_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        """Extract and project patch tokens: [B, 196, 128] L2-normalized."""
        with torch.inference_mode():
            tokens = self._forward_patch_tokens(x)
            flat = tokens.reshape(-1, self.VIT_EMBED_DIM)
            proj = self.proj(flat)
            proj = F.normalize(proj, p=2, dim=1)
        return proj.reshape(x.shape[0], self.PATCHES_PER_IMAGE, self.patch_embed_dim)

    def set_memory_bank(self, bank: torch.Tensor) -> None:
        normalized = F.normalize(bank.float(), p=2, dim=1)
        self.memory_bank = normalized.to(device=self.memory_bank.device)

    def nearest_patch_distances(self, patch_emb: torch.Tensor) -> torch.Tensor:
        """Cosine NN distance (euclidean on unit sphere) per patch: [B, 196]."""
        B, P, D = patch_emb.shape
        flat = patch_emb.reshape(-1, D)
        bank_t = self.memory_bank.t().contiguous()
        results = []
        for start in range(0, flat.shape[0], self.score_chunk):
            chunk = flat[start:start + self.score_chunk]
            sim = chunk @ bank_t
            k = min(self.nn_k, sim.shape[1])
            best = sim.topk(k=k, dim=1).values
            dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)
            results.append(dist)
        return torch.cat(results).reshape(B, P)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Image-level anomaly score (raw, before z-normalization)."""
        patch_emb = self.patch_embeddings(x)
        patch_dist = self.nearest_patch_distances(patch_emb)
        topk = max(1, int(round(self.PATCHES_PER_IMAGE * self.topk_patch_ratio)))
        return torch.topk(patch_dist, k=topk, dim=1).values.mean(dim=1)

def collect_memory_bank(model: ViTB16PatchCoreModel, loader: DataLoader, device: torch.device, target_size: int, seed: int) -> torch.Tensor:
    set_seed(seed)
    est_total = model.PATCHES_PER_IMAGE * len(loader.dataset)
    ratio = min(1.0, target_size / est_total)
    gen = torch.Generator().manual_seed(seed)
    batches = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in tqdm(loader, desc='memory_bank'):
            inputs = inputs.to(device)
            normal = inputs[labels == 0]
            if len(normal) == 0:
                continue
            emb = model.patch_embeddings(normal).reshape(-1, model.patch_embed_dim).cpu()
            if ratio < 1.0:
                keep = max(1, int(round(emb.shape[0] * ratio)))
                emb = emb[torch.randperm(emb.shape[0], generator=gen)[:keep]]
            batches.append(emb)
    bank = torch.cat(batches, dim=0)
    if bank.shape[0] > target_size:
        bank = bank[torch.randperm(bank.shape[0], generator=gen)[:target_size]]
    print(f'Memory bank: {bank.shape[0]:,} patches x {bank.shape[1]} dims')
    return bank

def collect_raw_scores(model: ViTB16PatchCoreModel, loader: DataLoader, device: torch.device, desc: str) -> pd.DataFrame:
    rows = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in tqdm(loader, desc=desc):
            scores = model(inputs.to(device)).cpu().numpy()
            for s, l in zip(scores, labels.numpy()):
                rows.append({'score_raw': float(s), 'is_anomaly': int(l)})
    return pd.DataFrame(rows)

def z_normalize_scores(raw_val_df: pd.DataFrame, raw_test_df: pd.DataFrame, quantile: float=0.95) -> tuple[pd.DataFrame, pd.DataFrame, float, float, float]:
    val_normal = raw_val_df.loc[raw_val_df['is_anomaly'] == 0, 'score_raw'].to_numpy()
    z_mean = float(val_normal.mean())
    z_std = float(val_normal.std())
    if z_std < 1e-08:
        z_std = 1.0

    def _znorm(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out['score'] = (out['score_raw'] - z_mean) / z_std
        return out
    val_z = _znorm(raw_val_df)
    test_z = _znorm(raw_test_df)
    val_z_normal = val_z.loc[val_z['is_anomaly'] == 0, 'score'].to_numpy()
    threshold_z = float(np.percentile(val_z_normal, quantile * 100))
    return (val_z, test_z, threshold_z, z_mean, z_std)

def make_loaders(dataset: dict, config: dict, device: torch.device) -> dict[str, DataLoader]:
    bs = int(config['data']['batch_size'])
    nw = 0 if os.name == 'nt' else int(config['data']['num_workers'])
    sz = int(config['data']['image_size'])
    pin = device.type == 'cuda'
    kw = dict(batch_size=bs, num_workers=nw, pin_memory=pin)
    return {'train': DataLoader(ViTArrayDataset(dataset['train_x'], dataset['train_y'], sz), shuffle=False, **kw), 'val': DataLoader(ViTArrayDataset(dataset['val_x'], dataset['val_y'], sz), shuffle=False, **kw), 'test': DataLoader(ViTArrayDataset(dataset['test_x'], dataset['test_y'], sz), shuffle=False, **kw)}

def compute_threshold_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum())
    tn = int(((pred == 0) & (labels == 0)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    return {'threshold': float(threshold), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1), 'roc_auc': float(roc_auc_score(labels, scores)), 'avg_precision': float(average_precision_score(labels, scores)), 'confusion_matrix': [[tn, fp], [fn, tp]]}

def sweep_percentile_thresholds(val_normal_z: np.ndarray, test_labels: np.ndarray, test_scores_z: np.ndarray) -> pd.DataFrame:
    rows = []
    for pct in np.arange(80, 100.1, 0.1):
        thr = float(np.percentile(val_normal_z, pct))
        pred = (test_scores_z >= thr).astype(int)
        tp = int(((pred == 1) & (test_labels == 1)).sum())
        fp = int(((pred == 1) & (test_labels == 0)).sum())
        fn = int(((pred == 0) & (test_labels == 1)).sum())
        prec = tp / max(1, tp + fp)
        rec = tp / max(1, tp + fn)
        f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
        rows.append({'percentile': round(pct, 1), 'threshold_z': thr, 'precision': prec, 'recall': rec, 'f1': f1})
    return pd.DataFrame(rows)

def build_defect_breakdown(defect_types: np.ndarray, labels: np.ndarray, scores: np.ndarray, threshold: float) -> pd.DataFrame:
    df = pd.DataFrame({'failure_label': defect_types, 'is_anomaly': labels.astype(int), 'score': scores})
    df['predicted'] = (df['score'] >= threshold).astype(int)
    defects = df[df['is_anomaly'] == 1]
    g = defects.groupby('failure_label').agg(count=('failure_label', 'size'), detected=('predicted', 'sum'), mean_score=('score', 'mean'))
    g['recall'] = g['detected'] / g['count'].clip(lower=1)
    return g.sort_values('recall', ascending=False).reset_index()
pd.DataFrame([CONFIG['data']]).T.rename(columns={0: 'value'})


In [ ]:
try:
    if RETRAIN:
        _raw_candidates = [REPO_ROOT / CONFIG['run']['raw_pickle'], REPO_ROOT / 'data' / 'raw' / 'LSWMD.pkl']
        _raw_path = next((p for p in _raw_candidates if p.exists()), None)
        if _raw_path is None:
            raise FileNotFoundError(f'LSWMD.pkl not found. Tried: {[str(p) for p in _raw_candidates]}')
        set_seed(CONFIG['run']['seed'])
        dataset = build_split_from_raw(raw_path=_raw_path, image_size=int(CONFIG['data']['image_size']), train_n=int(CONFIG['data']['train_normal_count']), val_n=int(CONFIG['data']['val_normal_count']), test_normal_n=int(CONFIG['data']['test_normal_count']), test_defect_n=int(CONFIG['data']['test_defect_count']), seed=int(CONFIG['run']['seed']))
        print('Dataset ready.')
    else:
        dataset = None
        print('RETRAIN=False: skipping dataset build. Loading artifacts from disk.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if retrain:\n    _raw_candidates = [repo_root / config['run']['raw_pickle'], repo_root / 'data' / 'raw' / 'lswmd.pkl']\n    _raw_path = next((p for p in _raw_candidates if p.exists()), none)\n    if _raw_path is none:\n        raise filenotfounderror(f'lswmd.pkl not found. tried: {[str(p) for p in _raw_candidates]}')\n    set_seed(config['run']['seed'])\n    dataset = build_split_from_raw(raw_path=_raw_path, image_size=int(config['data']['image_size']), train_n=int(config['data']['train_normal_count']), val_n=int(config['data']['val_normal_count']), test_normal_n=int(config['data']['test_normal_count']), test_defect_n=int(config['data']['test_defect_count']), seed=int(config['run']['seed']))\n    print('dataset ready.')\nelse:\n    dataset = none\n    print('retrain=false: skipping dataset build. loading artifacts from disk.')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    if RETRAIN:
        assert dataset is not None, 'dataset must be prepared before retraining'
        device = resolve_device()
        set_seed(CONFIG['run']['seed'])
        loaders = make_loaders(dataset, CONFIG, device)
        model = ViTB16PatchCoreModel(feature_block=int(CONFIG['model']['vit_feature_block']), patch_embed_dim=int(CONFIG['model']['patch_embed_dim']), nn_k=int(CONFIG['model']['patchcore_nn_k']), topk_patch_ratio=float(CONFIG['model']['topk_patch_ratio']), score_chunk=int(CONFIG['model']['score_chunk'])).to(device).eval()
        print(f'Device: {device}')
        bank = collect_memory_bank(model=model, loader=loaders['train'], device=device, target_size=int(CONFIG['model']['memory_bank_max_patches']), seed=int(CONFIG['run']['seed']))
        model.set_memory_bank(bank.to(device))
        raw_val_df = collect_raw_scores(model, loaders['val'], device, 'score_val')
        raw_test_df = collect_raw_scores(model, loaders['test'], device, 'score_test')
        val_scores_df, test_scores_df, threshold_z, z_mean, z_std = z_normalize_scores(raw_val_df, raw_test_df, quantile=float(CONFIG['scoring']['threshold_quantile']))
        threshold_raw = float(np.percentile(raw_val_df.loc[raw_val_df['is_anomaly'] == 0, 'score_raw'].to_numpy(), float(CONFIG['scoring']['threshold_quantile']) * 100))
        labels_z = test_scores_df['is_anomaly'].to_numpy()
        scores_z = test_scores_df['score'].to_numpy()
        metrics = compute_threshold_metrics(labels_z, scores_z, threshold_z)
        val_normal_z = val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score'].to_numpy()
        sweep_df = sweep_percentile_thresholds(val_normal_z, labels_z, scores_z)
        breakdown = build_defect_breakdown(dataset['test_defect_types'], labels_z, scores_z, threshold_z)
        best_sweep = sweep_df.sort_values('f1', ascending=False).iloc[0].to_dict()
        summary = {'evaluation_mode': 'main_5pct', 'variant_name': CONFIG['run']['variant_name'], 'backbone': CONFIG['model']['backbone'], 'threshold_policy': 'tune_normal_quantile_zscore', 'threshold_quantile': float(CONFIG['scoring']['threshold_quantile']), 'threshold_z': float(threshold_z), 'threshold_raw': float(threshold_raw), 'z_mean': float(z_mean), 'z_std': float(z_std), 'roc_auc_z': float(metrics['roc_auc']), 'avg_precision_z': float(metrics['avg_precision']), 'anomaly_precision': float(metrics['precision']), 'anomaly_recall': float(metrics['recall']), 'anomaly_f1': float(metrics['f1']), 'n_test_normal': int((labels_z == 0).sum()), 'n_test_defect': int((labels_z == 1).sum()), 'threshold_sweep_best': {'percentile': float(best_sweep['percentile']), 'threshold_z': float(best_sweep['threshold_z']), 'precision': float(best_sweep['precision']), 'recall': float(best_sweep['recall']), 'f1': float(best_sweep['f1'])}}
        (OUTPUT_DIR / 'results').mkdir(parents=True, exist_ok=True)
        EVAL_DIR.mkdir(parents=True, exist_ok=True)
        CHKPT_DIR.mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / 'results' / 'summary.json').write_text(json.dumps(summary, indent=2))
        val_scores_df.to_csv(EVAL_DIR / 'val_scores.csv', index=False)
        test_scores_df.to_csv(EVAL_DIR / 'test_scores.csv', index=False)
        sweep_df.to_csv(EVAL_DIR / 'threshold_sweep.csv', index=False)
        breakdown.to_csv(EVAL_DIR / 'saved_defect_breakdown.csv', index=False)
        torch.save({'model_state_dict': model.state_dict(), 'memory_bank': model.memory_bank.cpu(), 'z_mean': z_mean, 'z_std': z_std, 'threshold_z': threshold_z, 'threshold_raw': threshold_raw, 'summary': summary, 'config': CONFIG}, CHKPT_DIR / 'best_model.pt')
        print(f'Saved artifacts to {OUTPUT_DIR}')
    else:
        _summary_path = OUTPUT_DIR / 'results' / 'summary.json'
        _chkpt_path = CHKPT_DIR / 'best_model.pt'
        if not _summary_path.exists():
            raise FileNotFoundError(f'No summary at {_summary_path}. ')
        summary = json.loads(_summary_path.read_text())
        val_scores_df = pd.read_csv(EVAL_DIR / 'val_scores.csv')
        test_scores_df = pd.read_csv(EVAL_DIR / 'test_scores.csv')
        sweep_df = pd.read_csv(EVAL_DIR / 'threshold_sweep.csv')
        breakdown = pd.read_csv(EVAL_DIR / 'saved_defect_breakdown.csv')
        threshold_z = float(summary['threshold_z'])
        threshold_raw = float(summary['threshold_raw'])
        best_sweep = summary.get('threshold_sweep_best', {})
        labels_z = test_scores_df['is_anomaly'].to_numpy()
        scores_z = test_scores_df['score'].to_numpy()
        metrics = compute_threshold_metrics(labels_z, scores_z, threshold_z)
        print(f'Loaded artifacts from {OUTPUT_DIR}')
    print(f'\nthreshold_z  = {threshold_z:.4f}  (raw cosine: {threshold_raw:.4f})')
    print(f"F1={metrics['f1']:.3f}  AUROC={metrics['roc_auc']:.3f}  AUPRC={metrics['avg_precision']:.3f}")
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if retrain:\n    assert dataset is not none, 'dataset must be prepared before retraining'\n    device = resolve_device()\n    set_seed(config['run']['seed'])\n    loaders = make_loaders(dataset, config, device)\n    model = vitb16patchcoremodel(feature_block=int(config['model']['vit_feature_block']), patch_embed_dim=int(config['model']['patch_embed_dim']), nn_k=int(config['model']['patchcore_nn_k']), topk_patch_ratio=float(config['model']['topk_patch_ratio']), score_chunk=int(config['model']['score_chunk'])).to(device).eval()\n    print(f'device: {device}')\n    bank = collect_memory_bank(model=model, loader=loaders['train'], device=device, target_size=int(config['model']['memory_bank_max_patches']), seed=int(config['run']['seed']))\n    model.set_memory_bank(bank.to(device))\n    raw_val_df = collect_raw_scores(model, loaders['val'], device, 'score_val')\n    raw_test_df = collect_raw_scores(model, loaders['test'], device, 'score_test')\n    val_scores_df, test_scores_df, threshold_z, z_mean, z_std = z_normalize_scores(raw_val_df, raw_test_df, quantile=float(config['scoring']['threshold_quantile']))\n    threshold_raw = float(np.percentile(raw_val_df.loc[raw_val_df['is_anomaly'] == 0, 'score_raw'].to_numpy(), float(config['scoring']['threshold_quantile']) * 100))\n    labels_z = test_scores_df['is_anomaly'].to_numpy()\n    scores_z = test_scores_df['score'].to_numpy()\n    metrics = compute_threshold_metrics(labels_z, scores_z, threshold_z)\n    val_normal_z = val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score'].to_numpy()\n    sweep_df = sweep_percentile_thresholds(val_normal_z, labels_z, scores_z)\n    breakdown = build_defect_breakdown(dataset['test_defect_types'], labels_z, scores_z, threshold_z)\n    best_sweep = sweep_df.sort_values('f1', ascending=false).iloc[0].to_dict()\n    summary = {'evaluation_mode': 'main_5pct', 'variant_name': config['run']['variant_name'], 'backbone': config['model']['backbone'], 'threshold_policy': 'tune_normal_quantile_zscore', 'threshold_quantile': float(config['scoring']['threshold_quantile']), 'threshold_z': float(threshold_z), 'threshold_raw': float(threshold_raw), 'z_mean': float(z_mean), 'z_std': float(z_std), 'roc_auc_z': float(metrics['roc_auc']), 'avg_precision_z': float(metrics['avg_precision']), 'anomaly_precision': float(metrics['precision']), 'anomaly_recall': float(metrics['recall']), 'anomaly_f1': float(metrics['f1']), 'n_test_normal': int((labels_z == 0).sum()), 'n_test_defect': int((labels_z == 1).sum()), 'threshold_sweep_best': {'percentile': float(best_sweep['percentile']), 'threshold_z': float(best_sweep['threshold_z']), 'precision': float(best_sweep['precision']), 'recall': float(best_sweep['recall']), 'f1': float(best_sweep['f1'])}}\n    (output_dir / 'results').mkdir(parents=true, exist_ok=true)\n    eval_dir.mkdir(parents=true, exist_ok=true)\n    chkpt_dir.mkdir(parents=true, exist_ok=true)\n    (output_dir / 'results' / 'summary.json').write_text(json.dumps(summary, indent=2))\n    val_scores_df.to_csv(eval_dir / 'val_scores.csv', index=false)\n    test_scores_df.to_csv(eval_dir / 'test_scores.csv', index=false)\n    sweep_df.to_csv(eval_dir / 'threshold_sweep.csv', index=false)\n    breakdown.to_csv(eval_dir / 'saved_defect_breakdown.csv', index=false)\n    torch.save({'model_state_dict': model.state_dict(), 'memory_bank': model.memory_bank.cpu(), 'z_mean': z_mean, 'z_std': z_std, 'threshold_z': threshold_z, 'threshold_raw': threshold_raw, 'summary': summary, 'config': config}, chkpt_dir / 'best_model.pt')\n    print(f'saved artifacts to {output_dir}')\nelse:\n    _summary_path = output_dir / 'results' / 'summary.json'\n    _chkpt_path = chkpt_dir / 'best_model.pt'\n    if not _summary_path.exists():\n        raise filenotfounderror(f'no summary at {_summary_path}. ')\n    summary = json.loads(_summary_path.read_text())\n    val_scores_df = pd.read_csv(eval_dir / 'val_scores.csv')\n    test_scores_df = pd.read_csv(eval_dir / 'test_scores.csv')\n    sweep_df = pd.read_csv(eval_dir / 'threshold_sw"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    if RETRAIN:
        import umap.umap_ as umap_lib
        from sklearn.decomposition import PCA
        from sklearn.neighbors import NearestNeighbors
        UMAP_DIR.mkdir(parents=True, exist_ok=True)
        _uc = CONFIG['umap']
        _max_ref = int(_uc['max_train_reference'])
        _max_valn = int(_uc['max_val_normal'])
        _max_testn = int(_uc['max_test_normal'])
        _max_testa = int(_uc['max_test_anomaly'])
        _pca_k = int(_uc['pca_components'])
        _knn_k = int(_uc['knn_k'])
        _seed = int(CONFIG['run']['seed'])

        def _collect_embeddings(mdl, ldr, dev, tag):
            embs, labs, raw = ([], [], [])
            mdl.eval()
            with torch.inference_mode():
                for x, y in tqdm(ldr, desc=tag):
                    x = x.to(dev)
                    pe = mdl.patch_embeddings(x)
                    embs.append(pe.mean(dim=1).cpu().numpy())
                    labs.extend(y.tolist())
                    raw.extend(mdl(x).cpu().tolist())
            return (np.concatenate(embs), np.array(labs), np.array(raw))
        print('Collecting embeddings for UMAP..')
        tr_emb, tr_lbl, _ = _collect_embeddings(model, loaders['train'], device, 'umap_train')
        vl_emb, vl_lbl, vl_raw = _collect_embeddings(model, loaders['val'], device, 'umap_val')
        ts_emb, ts_lbl, ts_raw = _collect_embeddings(model, loaders['test'], device, 'umap_test')
        vl_z = (vl_raw - z_mean) / z_std
        ts_z = (ts_raw - z_mean) / z_std
        rng = np.random.default_rng(_seed)
        ref_idx = rng.choice(np.where(tr_lbl == 0)[0], size=min(_max_ref, (tr_lbl == 0).sum()), replace=False)
        valn_idx = rng.choice(np.where(vl_lbl == 0)[0], size=min(_max_valn, (vl_lbl == 0).sum()), replace=False)
        tsn_idx = rng.choice(np.where(ts_lbl == 0)[0], size=min(_max_testn, (ts_lbl == 0).sum()), replace=False)
        tsa_idx = rng.choice(np.where(ts_lbl == 1)[0], size=min(_max_testa, (ts_lbl == 1).sum()), replace=False)
        ref_emb = tr_emb[ref_idx]
        valn_emb = vl_emb[valn_idx]
        tsn_emb = ts_emb[tsn_idx]
        tsa_emb = ts_emb[tsa_idx]
        _n_pca = min(_pca_k, ref_emb.shape[1], ref_emb.shape[0] - 1)
        pca = PCA(n_components=_n_pca, random_state=_seed)
        ref_pca = pca.fit_transform(ref_emb)
        valn_pca = pca.transform(valn_emb)
        tsn_pca = pca.transform(tsn_emb)
        tsa_pca = pca.transform(tsa_emb)
        print('Fitting UMAP (cosine metric, train-reference only)..')
        reducer = umap_lib.UMAP(n_components=2, n_neighbors=int(_uc['n_neighbors']), min_dist=float(_uc['min_dist']), metric=str(_uc['metric']), random_state=_seed)
        ref_2d = reducer.fit_transform(ref_pca)
        valn_2d = reducer.transform(valn_pca)
        tsn_2d = reducer.transform(tsn_pca)
        tsa_2d = reducer.transform(tsa_pca)
        knn = NearestNeighbors(n_neighbors=_knn_k, metric='euclidean')
        knn.fit(ref_2d)
        valn_knn = knn.kneighbors(valn_2d)[0].mean(axis=1)
        tsn_knn = knn.kneighbors(tsn_2d)[0].mean(axis=1)
        tsa_knn = knn.kneighbors(tsa_2d)[0].mean(axis=1)
        umap_thr = float(np.percentile(valn_knn, float(CONFIG['scoring']['threshold_quantile']) * 100))
        _all_knn = np.concatenate([tsn_knn, tsa_knn])
        _all_lbl = np.concatenate([np.zeros(len(tsn_knn)), np.ones(len(tsa_knn))])
        umap_knn_metrics = compute_threshold_metrics(_all_lbl, _all_knn, umap_thr)
        _sweep_rows = []
        for _t in np.linspace(_all_knn.min(), _all_knn.max(), 200):
            _pred = (_all_knn >= _t).astype(int)
            _tp = int(((_pred == 1) & (_all_lbl == 1)).sum())
            _fp = int(((_pred == 1) & (_all_lbl == 0)).sum())
            _fn = int(((_pred == 0) & (_all_lbl == 1)).sum())
            _p = _tp / max(1, _tp + _fp)
            _r = _tp / max(1, _tp + _fn)
            _f = 0.0 if _p + _r == 0 else 2 * _p * _r / (_p + _r)
            _sweep_rows.append({'threshold': float(_t), 'precision': _p, 'recall': _r, 'f1': _f})
        umap_sweep_df = pd.DataFrame(_sweep_rows)
        _best_umap = umap_sweep_df.sort_values('f1', ascending=False).iloc[0].to_dict()
        _rows = []
        for i, (ux, uy) in enumerate(ref_2d):
            _rows.append({'umap_1': ux, 'umap_2': uy, 'split_label': 'train_reference', 'is_anomaly': 0, 'model_score': None, 'umap_knn_score': None})
        for i, (ux, uy) in enumerate(valn_2d):
            _rows.append({'umap_1': ux, 'umap_2': uy, 'split_label': 'val_normal', 'is_anomaly': 0, 'model_score': float(vl_z[valn_idx[i]]), 'umap_knn_score': float(valn_knn[i])})
        for i, (ux, uy) in enumerate(tsn_2d):
            _rows.append({'umap_1': ux, 'umap_2': uy, 'split_label': 'test_normal', 'is_anomaly': 0, 'model_score': float(ts_z[tsn_idx[i]]), 'umap_knn_score': float(tsn_knn[i])})
        for i, (ux, uy) in enumerate(tsa_2d):
            _rows.append({'umap_1': ux, 'umap_2': uy, 'split_label': 'test_anomaly', 'is_anomaly': 1, 'model_score': float(ts_z[tsa_idx[i]]), 'umap_knn_score': float(tsa_knn[i])})
        umap_df = pd.DataFrame(_rows)
        umap_df.to_csv(UMAP_DIR / 'umap_test_embeddings.csv', index=False)
        umap_summary_data = {'threshold_quantile': float(CONFIG['scoring']['threshold_quantile']), 'counts': {'train_reference': len(ref_2d), 'val_total': len(valn_2d), 'val_normal': len(valn_2d), 'val_anomaly': 0, 'test_total': len(tsn_2d) + len(tsa_2d), 'test_normal': len(tsn_2d), 'test_anomaly': len(tsa_2d)}, 'plot_counts': {'train_reference': len(ref_2d), 'val_normal': len(valn_2d), 'val_anomaly': 0, 'test_normal': len(tsn_2d), 'test_anomaly': len(tsa_2d)}, 'umap_params': {'random_state': _seed, 'pca_components': _n_pca, 'n_neighbors': int(_uc['n_neighbors']), 'min_dist': float(_uc['min_dist']), 'metric': str(_uc['metric']), 'knn_k': _knn_k}, 'umap_knn_threshold': float(umap_thr), 'umap_knn_metrics': umap_knn_metrics, 'umap_knn_best_threshold_sweep': _best_umap}
        (UMAP_DIR / 'umap_summary.json').write_text(json.dumps(umap_summary_data, indent=2))
        umap_sweep_df.to_csv(UMAP_DIR / 'umap_knn_threshold_sweep.csv', index=False)
        _plot = umap_df[umap_df['split_label'].isin(['val_normal', 'test_normal', 'test_anomaly'])]
        _is_a = (_plot['split_label'] == 'test_anomaly').values
        _ux, _uy = (_plot['umap_1'].values, _plot['umap_2'].values)
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(_ux[~_is_a], _uy[~_is_a], s=3, alpha=0.15, c='#aaaaaa', linewidths=0, label='Normal')
        ax.scatter(_ux[_is_a], _uy[_is_a], s=20, alpha=0.85, c='#e41a1c', linewidths=0, label='Anomaly')
        ax.set_title('ViT-B/16 PatchCore UMAP -- Split-coloured (cosine metric)', fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / 'umap_test_embeddings.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(7, 6))
        sc = ax.scatter(_ux, _uy, c=_plot['model_score'].values.astype(float), cmap='hot_r', s=3, alpha=0.6, linewidths=0)
        plt.colorbar(sc, ax=ax, label='Anomaly score (z-normalised)')
        ax.set_title('ViT-B/16 PatchCore UMAP -- Score-coloured (cosine metric)', fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / 'umap_by_score.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f'UMAP artifacts saved to {UMAP_DIR}')
        display(pd.DataFrame([{'umap_knn_threshold': round(umap_thr, 4), 'umap_knn_f1': round(umap_knn_metrics['f1'], 4), 'umap_knn_auroc': round(umap_knn_metrics['roc_auc'], 4), 'n_train_reference': len(ref_2d), 'n_test_normal': len(tsn_2d), 'n_test_anomaly': len(tsa_2d)}]))
    else:
        print('UMAP generation skipped (RETRAIN=False).')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if retrain:\n    import umap.umap_ as umap_lib\n    from sklearn.decomposition import pca\n    from sklearn.neighbors import nearestneighbors\n    umap_dir.mkdir(parents=true, exist_ok=true)\n    _uc = config['umap']\n    _max_ref = int(_uc['max_train_reference'])\n    _max_valn = int(_uc['max_val_normal'])\n    _max_testn = int(_uc['max_test_normal'])\n    _max_testa = int(_uc['max_test_anomaly'])\n    _pca_k = int(_uc['pca_components'])\n    _knn_k = int(_uc['knn_k'])\n    _seed = int(config['run']['seed'])\n\n    def _collect_embeddings(mdl, ldr, dev, tag):\n        embs, labs, raw = ([], [], [])\n        mdl.eval()\n        with torch.inference_mode():\n            for x, y in tqdm(ldr, desc=tag):\n                x = x.to(dev)\n                pe = mdl.patch_embeddings(x)\n                embs.append(pe.mean(dim=1).cpu().numpy())\n                labs.extend(y.tolist())\n                raw.extend(mdl(x).cpu().tolist())\n        return (np.concatenate(embs), np.array(labs), np.array(raw))\n    print('collecting embeddings for umap..')\n    tr_emb, tr_lbl, _ = _collect_embeddings(model, loaders['train'], device, 'umap_train')\n    vl_emb, vl_lbl, vl_raw = _collect_embeddings(model, loaders['val'], device, 'umap_val')\n    ts_emb, ts_lbl, ts_raw = _collect_embeddings(model, loaders['test'], device, 'umap_test')\n    vl_z = (vl_raw - z_mean) / z_std\n    ts_z = (ts_raw - z_mean) / z_std\n    rng = np.random.default_rng(_seed)\n    ref_idx = rng.choice(np.where(tr_lbl == 0)[0], size=min(_max_ref, (tr_lbl == 0).sum()), replace=false)\n    valn_idx = rng.choice(np.where(vl_lbl == 0)[0], size=min(_max_valn, (vl_lbl == 0).sum()), replace=false)\n    tsn_idx = rng.choice(np.where(ts_lbl == 0)[0], size=min(_max_testn, (ts_lbl == 0).sum()), replace=false)\n    tsa_idx = rng.choice(np.where(ts_lbl == 1)[0], size=min(_max_testa, (ts_lbl == 1).sum()), replace=false)\n    ref_emb = tr_emb[ref_idx]\n    valn_emb = vl_emb[valn_idx]\n    tsn_emb = ts_emb[tsn_idx]\n    tsa_emb = ts_emb[tsa_idx]\n    _n_pca = min(_pca_k, ref_emb.shape[1], ref_emb.shape[0] - 1)\n    pca = pca(n_components=_n_pca, random_state=_seed)\n    ref_pca = pca.fit_transform(ref_emb)\n    valn_pca = pca.transform(valn_emb)\n    tsn_pca = pca.transform(tsn_emb)\n    tsa_pca = pca.transform(tsa_emb)\n    print('fitting umap (cosine metric, train-reference only)..')\n    reducer = umap_lib.umap(n_components=2, n_neighbors=int(_uc['n_neighbors']), min_dist=float(_uc['min_dist']), metric=str(_uc['metric']), random_state=_seed)\n    ref_2d = reducer.fit_transform(ref_pca)\n    valn_2d = reducer.transform(valn_pca)\n    tsn_2d = reducer.transform(tsn_pca)\n    tsa_2d = reducer.transform(tsa_pca)\n    knn = nearestneighbors(n_neighbors=_knn_k, metric='euclidean')\n    knn.fit(ref_2d)\n    valn_knn = knn.kneighbors(valn_2d)[0].mean(axis=1)\n    tsn_knn = knn.kneighbors(tsn_2d)[0].mean(axis=1)\n    tsa_knn = knn.kneighbors(tsa_2d)[0].mean(axis=1)\n    umap_thr = float(np.percentile(valn_knn, float(config['scoring']['threshold_quantile']) * 100))\n    _all_knn = np.concatenate([tsn_knn, tsa_knn])\n    _all_lbl = np.concatenate([np.zeros(len(tsn_knn)), np.ones(len(tsa_knn))])\n    umap_knn_metrics = compute_threshold_metrics(_all_lbl, _all_knn, umap_thr)\n    _sweep_rows = []\n    for _t in np.linspace(_all_knn.min(), _all_knn.max(), 200):\n        _pred = (_all_knn >= _t).astype(int)\n        _tp = int(((_pred == 1) & (_all_lbl == 1)).sum())\n        _fp = int(((_pred == 1) & (_all_lbl == 0)).sum())\n        _fn = int(((_pred == 0) & (_all_lbl == 1)).sum())\n        _p = _tp / max(1, _tp + _fp)\n        _r = _tp / max(1, _tp + _fn)\n        _f = 0.0 if _p + _r == 0 else 2 * _p * _r / (_p + _r)\n        _sweep_rows.append({'threshold': float(_t), 'precision': _p, 'recall': _r, 'f1': _f})\n    umap_sweep_df = pd.dataframe(_sweep_rows)\n    _best_umap = umap_sweep_df.sort_values('f1', ascending=false).iloc[0].to_dict()\n    _rows = []\n    for i, (ux, uy) in enumerate(ref_2d):\n        _rows.append({'umap_1': ux, 'umap_2': uy, 'split_label': 'train_referen"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    display(pd.DataFrame([{'variant': summary.get('variant_name', 'vit_b16_one_layer_patchcore_x224'), 'backbone': summary.get('backbone', 'vit_base_patch16_224.augreg_in21k_ft_in1k'), 'threshold_z': round(threshold_z, 4), 'threshold_raw': round(threshold_raw, 4), 'precision': round(metrics['precision'], 4), 'recall': round(metrics['recall'], 4), 'f1': round(metrics['f1'], 4), 'roc_auc': round(metrics['roc_auc'], 4), 'avg_precision': round(metrics['avg_precision'], 4), 'n_test_normal': int((labels_z == 0).sum()), 'n_test_defect': int((labels_z == 1).sum())}]))
    display(pd.DataFrame(metrics['confusion_matrix'], index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly']))
    display(breakdown)
    _val_normal_scores = val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score'].to_numpy()
    _test_normal_scores = test_scores_df.loc[test_scores_df['is_anomaly'] == 0, 'score'].to_numpy()
    _test_anomaly_scores = test_scores_df.loc[test_scores_df['is_anomaly'] == 1, 'score'].to_numpy()
    _best_thresh = float(best_sweep.get('threshold_z', threshold_z))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(_val_normal_scores, bins=40, alpha=0.55, color='#3d405b', label='val normal')
    ax.hist(_test_normal_scores, bins=40, alpha=0.55, color='#577590', label='test normal')
    ax.hist(_test_anomaly_scores, bins=40, alpha=0.75, color='#e76f51', label='test anomaly')
    ax.axvline(threshold_z, color='red', linestyle='--', linewidth=1.5, label=f'deployed (95th pct) = {threshold_z:.4f}')
    ax.axvline(_best_thresh, color='green', linestyle=':', linewidth=1.5, label=f'best-sweep = {_best_thresh:.4f}')
    ax.set_xlabel('Anomaly score (z-normalised cosine NN distance, top-10% patches)')
    ax.set_ylabel('Count')
    ax.set_title('ViT-B/16 PatchCore -- Nearest-Neighbour Distance Distribution')
    ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'nn_distance_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color='#3d405b')
    axes[0].axvline(threshold_z, color='red', linestyle='--')
    axes[0].set_title('Validation Score Distribution')
    axes[1].hist(_test_normal_scores, bins=30, alpha=0.7, label='normal', color='#577590')
    axes[1].hist(_test_anomaly_scores, bins=30, alpha=0.7, label='anomaly', color='#e76f51')
    axes[1].axvline(threshold_z, color='red', linestyle='--')
    axes[1].set_title('Test Score Distribution')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    if 'percentile' in sweep_df.columns:
        _sweep_x = sweep_df['percentile']
    else:
        _sweep_x = sweep_df.index
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(_sweep_x, sweep_df['precision'], label='precision')
    ax.plot(_sweep_x, sweep_df['recall'], label='recall')
    ax.plot(_sweep_x, sweep_df['f1'], label='f1')
    ax.set_xlabel('Percentile of val-normal score')
    ax.set_title('Threshold Sweep by Percentile')
    ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    if 'recall' in breakdown.columns and 'failure_label' in breakdown.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        _bd = breakdown.sort_values('recall', ascending=False)
        ax.barh(_bd['failure_label'], _bd['recall'], color='#81b29a')
        ax.set_xlim(0.0, 1.0)
        ax.invert_yaxis()
        ax.set_title('Defect Recall by Failure Label')
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "display(pd.dataframe([{'variant': summary.get('variant_name', 'vit_b16_one_layer_patchcore_x224'), 'backbone': summary.get('backbone', 'vit_base_patch16_224.augreg_in21k_ft_in1k'), 'threshold_z': round(threshold_z, 4), 'threshold_raw': round(threshold_raw, 4), 'precision': round(metrics['precision'], 4), 'recall': round(metrics['recall'], 4), 'f1': round(metrics['f1'], 4), 'roc_auc': round(metrics['roc_auc'], 4), 'avg_precision': round(metrics['avg_precision'], 4), 'n_test_normal': int((labels_z == 0).sum()), 'n_test_defect': int((labels_z == 1).sum())}]))\ndisplay(pd.dataframe(metrics['confusion_matrix'], index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly']))\ndisplay(breakdown)\n_val_normal_scores = val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score'].to_numpy()\n_test_normal_scores = test_scores_df.loc[test_scores_df['is_anomaly'] == 0, 'score'].to_numpy()\n_test_anomaly_scores = test_scores_df.loc[test_scores_df['is_anomaly'] == 1, 'score'].to_numpy()\n_best_thresh = float(best_sweep.get('threshold_z', threshold_z))\nfig, ax = plt.subplots(figsize=(9, 4))\nax.hist(_val_normal_scores, bins=40, alpha=0.55, color='#3d405b', label='val normal')\nax.hist(_test_normal_scores, bins=40, alpha=0.55, color='#577590', label='test normal')\nax.hist(_test_anomaly_scores, bins=40, alpha=0.75, color='#e76f51', label='test anomaly')\nax.axvline(threshold_z, color='red', linestyle='--', linewidth=1.5, label=f'deployed (95th pct) = {threshold_z:.4f}')\nax.axvline(_best_thresh, color='green', linestyle=':', linewidth=1.5, label=f'best-sweep = {_best_thresh:.4f}')\nax.set_xlabel('anomaly score (z-normalised cosine nn distance, top-10% patches)')\nax.set_ylabel('count')\nax.set_title('vit-b/16 patchcore -- nearest-neighbour distance distribution')\nax.legend(fontsize=9)\nplt.tight_layout()\nfig.savefig(plots_dir / 'nn_distance_distribution.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)\nfig, axes = plt.subplots(1, 2, figsize=(12, 4))\naxes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color='#3d405b')\naxes[0].axvline(threshold_z, color='red', linestyle='--')\naxes[0].set_title('validation score distribution')\naxes[1].hist(_test_normal_scores, bins=30, alpha=0.7, label='normal', color='#577590')\naxes[1].hist(_test_anomaly_scores, bins=30, alpha=0.7, label='anomaly', color='#e76f51')\naxes[1].axvline(threshold_z, color='red', linestyle='--')\naxes[1].set_title('test score distribution')\naxes[1].legend()\nplt.tight_layout()\nfig.savefig(plots_dir / 'score_distribution.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)\nif 'percentile' in sweep_df.columns:\n    _sweep_x = sweep_df['percentile']\nelse:\n    _sweep_x = sweep_df.index\nfig, ax = plt.subplots(figsize=(8, 4))\nax.plot(_sweep_x, sweep_df['precision'], label='precision')\nax.plot(_sweep_x, sweep_df['recall'], label='recall')\nax.plot(_sweep_x, sweep_df['f1'], label='f1')\nax.set_xlabel('percentile of val-normal score')\nax.set_title('threshold sweep by percentile')\nax.legend()\nplt.tight_layout()\nfig.savefig(plots_dir / 'threshold_sweep.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)\nif 'recall' in breakdown.columns and 'failure_label' in breakdown.columns:\n    fig, ax = plt.subplots(figsize=(10, 5))\n    _bd = breakdown.sort_values('recall', ascending=false)\n    ax.barh(_bd['failure_label'], _bd['recall'], color='#81b29a')\n    ax.set_xlim(0.0, 1.0)\n    ax.invert_yaxis()\n    ax.set_title('defect recall by failure label')\n    plt.tight_layout()\n    fig.savefig(plots_dir / 'defect_breakdown.png', dpi=200, bbox_inches='tight')\n    plt.show()\n    plt.close(fig)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Holdout Evaluation

Secondary evaluation on the larger holdout set (70 k normals / 3.5 k defects).
Controlled by `RUN_HOLDOUT = True/False`.
When `RETRAIN = False`, artifacts are loaded from `holdout70k_3p5k/results/evaluation/`.


In [ ]:
try:
    _h_summary_path = HOLDOUT_DIR / 'results' / 'summary.json'
    _h_csvs_exist = _h_summary_path.exists() and (H_EVAL_DIR / 'val_scores.csv').exists()
    if RETRAIN and RUN_HOLDOUT:
        _raw_path = REPO_ROOT / CONFIG['run']['raw_pickle']
        set_seed(CONFIG['run']['seed'])
        holdout_dataset = build_split_from_raw(raw_path=_raw_path, image_size=int(CONFIG['data']['image_size']), train_n=int(CONFIG['data']['train_normal_count']), val_n=int(CONFIG['data']['val_normal_count']), test_normal_n=int(CONFIG['data']['holdout_normal_count']), test_defect_n=int(CONFIG['data']['holdout_defect_count']), seed=int(CONFIG['run']['seed']))
        holdout_loaders = make_loaders(holdout_dataset, CONFIG, device)
        _chkpt = torch.load(CHKPT_DIR / 'best_model.pt', map_location=device)
        _h_z_mean = float(_chkpt.get('z_mean', z_mean))
        _h_z_std = float(_chkpt.get('z_std', z_std))
        h_raw_val = collect_raw_scores(model, holdout_loaders['val'], device, 'h_score_val')
        h_raw_test = collect_raw_scores(model, holdout_loaders['test'], device, 'h_score_test')
        h_val_df = h_raw_val.copy()
        h_val_df['score'] = (h_raw_val['score_raw'] - _h_z_mean) / _h_z_std
        h_test_df = h_raw_test.copy()
        h_test_df['score'] = (h_raw_test['score_raw'] - _h_z_mean) / _h_z_std
        _h_val_normal_z = h_val_df.loc[h_val_df['is_anomaly'] == 0, 'score'].to_numpy()
        h_threshold_z = float(np.percentile(_h_val_normal_z, float(CONFIG['scoring']['threshold_quantile']) * 100))
        h_labels = h_test_df['is_anomaly'].to_numpy()
        h_scores = h_test_df['score'].to_numpy()
        h_metrics = compute_threshold_metrics(h_labels, h_scores, h_threshold_z)
        h_breakdown = build_defect_breakdown(holdout_dataset['test_defect_types'], h_labels, h_scores, h_threshold_z)
        h_sweep_df = sweep_percentile_thresholds(h_val_df.loc[h_val_df['is_anomaly'] == 0, 'score'].to_numpy(), h_labels, h_scores)
        _h_best_sweep = h_sweep_df.sort_values('f1', ascending=False).iloc[0].to_dict()
        H_EVAL_DIR.mkdir(parents=True, exist_ok=True)
        H_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        h_val_df.to_csv(H_EVAL_DIR / 'val_scores.csv', index=False)
        h_test_df.to_csv(H_EVAL_DIR / 'test_scores.csv', index=False)
        h_breakdown.to_csv(H_EVAL_DIR / 'saved_defect_breakdown.csv', index=False)
        h_sweep_df.to_csv(H_EVAL_DIR / 'threshold_sweep.csv', index=False)
        h_summary = {'evaluation_mode': 'holdout70k_3p5k', 'threshold_z': float(h_threshold_z), 'threshold_raw': float(h_threshold_z * _h_z_std + _h_z_mean), 'precision': float(h_metrics['precision']), 'recall': float(h_metrics['recall']), 'f1': float(h_metrics['f1']), 'roc_auc_z': float(h_metrics['roc_auc']), 'avg_precision_z': float(h_metrics['avg_precision']), 'n_test_normal': int((h_labels == 0).sum()), 'n_test_defect': int((h_labels == 1).sum()), 'threshold_sweep_best': _h_best_sweep}
        (HOLDOUT_DIR / 'results').mkdir(parents=True, exist_ok=True)
        (HOLDOUT_DIR / 'results' / 'summary.json').write_text(json.dumps(h_summary, indent=2))
        print(f'Holdout artifacts saved to {HOLDOUT_DIR}')
    elif _h_csvs_exist:
        h_summary = json.loads(_h_summary_path.read_text())
        h_val_df = pd.read_csv(H_EVAL_DIR / 'val_scores.csv')
        h_test_df = pd.read_csv(H_EVAL_DIR / 'test_scores.csv')
        h_sweep_df = pd.read_csv(H_EVAL_DIR / 'threshold_sweep.csv')
        h_breakdown = pd.read_csv(H_EVAL_DIR / 'saved_defect_breakdown.csv')
        h_threshold_z = float(h_summary['threshold_z'])
        h_labels = h_test_df['is_anomaly'].to_numpy()
        h_scores = h_test_df['score'].to_numpy()
        h_metrics = compute_threshold_metrics(h_labels, h_scores, h_threshold_z)
        _h_best = float(h_summary.get('threshold_sweep_best', {}).get('threshold_z', h_threshold_z))
        if not RUN_HOLDOUT:
            print('RUN_HOLDOUT=False but pre-saved holdout artifacts found - loading and plotting.')
    else:
        print('Holdout skipped: RUN_HOLDOUT=False and no pre-saved artifacts found.')
        h_summary = None
    if 'h_summary' in dir() and h_summary is not None:
        H_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        display(pd.DataFrame([{'protocol': 'holdout70k_3p5k', 'threshold_z': round(h_threshold_z, 4), 'precision': round(h_metrics['precision'], 4), 'recall': round(h_metrics['recall'], 4), 'f1': round(h_metrics['f1'], 4), 'roc_auc': round(h_metrics['roc_auc'], 4), 'avg_precision': round(h_metrics['avg_precision'], 4), 'n_test_normal': int((h_labels == 0).sum()), 'n_test_defect': int((h_labels == 1).sum())}]))
        display(h_breakdown)
        _h_valn = h_val_df.loc[h_val_df['is_anomaly'] == 0, 'score'].to_numpy()
        _h_testn = h_test_df.loc[h_test_df['is_anomaly'] == 0, 'score'].to_numpy()
        _h_testa = h_test_df.loc[h_test_df['is_anomaly'] == 1, 'score'].to_numpy()
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.hist(_h_valn, bins=40, alpha=0.55, color='#3d405b', label='val normal')
        ax.hist(_h_testn, bins=40, alpha=0.55, color='#577590', label='test normal')
        ax.hist(_h_testa, bins=40, alpha=0.75, color='#e76f51', label='test anomaly')
        ax.axvline(h_threshold_z, color='red', linestyle='--', linewidth=1.5, label=f'deployed = {h_threshold_z:.4f}')
        ax.axvline(_h_best, color='green', linestyle=':', linewidth=1.5, label=f'best-sweep = {_h_best:.4f}')
        ax.set_xlabel('Anomaly score (z-normalised)')
        ax.set_ylabel('Count')
        ax.set_title('Holdout -- Score Distribution')
        ax.legend(fontsize=8)
        plt.tight_layout()
        fig.savefig(H_PLOTS_DIR / 'score_distribution.png', dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        if 'percentile' in h_sweep_df.columns or 'threshold_z' in h_sweep_df.columns:
            _hsx = h_sweep_df['percentile'] if 'percentile' in h_sweep_df.columns else h_sweep_df.index
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(_hsx, h_sweep_df['precision'], label='precision')
            ax.plot(_hsx, h_sweep_df['recall'], label='recall')
            ax.plot(_hsx, h_sweep_df['f1'], label='f1')
            ax.set_xlabel('Percentile')
            ax.set_title('Holdout -- Threshold Sweep')
            ax.legend()
            plt.tight_layout()
            fig.savefig(H_PLOTS_DIR / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
        if 'recall' in h_breakdown.columns and 'failure_label' in h_breakdown.columns:
            fig, ax = plt.subplots(figsize=(10, 5))
            _hbd = h_breakdown.sort_values('recall', ascending=False)
            ax.barh(_hbd['failure_label'], _hbd['recall'], color='#81b29a')
            ax.set_xlim(0.0, 1.0)
            ax.invert_yaxis()
            ax.set_title('Holdout -- Defect Recall by Failure Label')
            plt.tight_layout()
            fig.savefig(H_PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
            plt.show()
            plt.close(fig)
        _h_split_png = H_PLOTS_DIR / 'umap_test_embeddings.png'
        _h_score_png = H_PLOTS_DIR / 'umap_by_score.png'
        _h_umap_csv = H_UMAP_DIR / 'umap_test_embeddings.csv'
        _h_need_replot = REGENERATE_UMAP or (not _h_split_png.exists() and _h_umap_csv.exists())
        if _h_need_replot and _h_umap_csv.exists():
            _hdf = pd.read_csv(_h_umap_csv)
            _hxc = 'umap_1' if 'umap_1' in _hdf.columns else 'umap1'
            _hyc = 'umap_2' if 'umap_2' in _hdf.columns else 'umap2'
            _hx, _hy = (_hdf[_hxc].values, _hdf[_hyc].values)
            _hsc = next((c for c in ['split_label', 'group'] if c in _hdf.columns), None)
            _hia = _hdf[_hsc].isin(['test_anomaly']).values if _hsc else _hdf['is_anomaly'].values.astype(bool) if 'is_anomaly' in _hdf.columns else np.zeros(len(_hdf), dtype=bool)
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.scatter(_hx[~_hia], _hy[~_hia], s=3, alpha=0.12, c='#aaaaaa', linewidths=0, label='Normal')
            ax.scatter(_hx[_hia], _hy[_hia], s=15, alpha=0.8, c='#e41a1c', linewidths=0, label='Anomaly')
            ax.set_title('Holdout UMAP -- Split-coloured', fontweight='bold')
            ax.legend(fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            plt.tight_layout()
            fig.savefig(_h_split_png, dpi=150, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            _hscol = next((c for c in ['score', 'model_score'] if c in _hdf.columns), None)
            if _hscol:
                _hhas = _hdf[_hscol].notna().values
                fig, ax = plt.subplots(figsize=(7, 6))
                sc = ax.scatter(_hx[_hhas], _hy[_hhas], c=_hdf[_hscol].values[_hhas].astype(float), cmap='hot_r', s=3, alpha=0.5, linewidths=0)
                plt.colorbar(sc, ax=ax, label='Anomaly score (z-normalised)')
                ax.set_title('Holdout UMAP -- Score-coloured', fontweight='bold')
                ax.set_xticks([])
                ax.set_yticks([])
                plt.tight_layout()
                fig.savefig(_h_score_png, dpi=150, bbox_inches='tight')
                plt.show()
                plt.close(fig)
        elif not _h_need_replot:
            for _png in [_h_split_png, _h_score_png]:
                if _png.exists():
                    print(f'Displaying: {_png.name}')
                    display(IPImage(filename=str(_png)))
        print(f'Holdout plots saved to {H_PLOTS_DIR}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "_h_summary_path = holdout_dir / 'results' / 'summary.json'\n_h_csvs_exist = _h_summary_path.exists() and (h_eval_dir / 'val_scores.csv').exists()\nif retrain and run_holdout:\n    _raw_path = repo_root / config['run']['raw_pickle']\n    set_seed(config['run']['seed'])\n    holdout_dataset = build_split_from_raw(raw_path=_raw_path, image_size=int(config['data']['image_size']), train_n=int(config['data']['train_normal_count']), val_n=int(config['data']['val_normal_count']), test_normal_n=int(config['data']['holdout_normal_count']), test_defect_n=int(config['data']['holdout_defect_count']), seed=int(config['run']['seed']))\n    holdout_loaders = make_loaders(holdout_dataset, config, device)\n    _chkpt = torch.load(chkpt_dir / 'best_model.pt', map_location=device)\n    _h_z_mean = float(_chkpt.get('z_mean', z_mean))\n    _h_z_std = float(_chkpt.get('z_std', z_std))\n    h_raw_val = collect_raw_scores(model, holdout_loaders['val'], device, 'h_score_val')\n    h_raw_test = collect_raw_scores(model, holdout_loaders['test'], device, 'h_score_test')\n    h_val_df = h_raw_val.copy()\n    h_val_df['score'] = (h_raw_val['score_raw'] - _h_z_mean) / _h_z_std\n    h_test_df = h_raw_test.copy()\n    h_test_df['score'] = (h_raw_test['score_raw'] - _h_z_mean) / _h_z_std\n    _h_val_normal_z = h_val_df.loc[h_val_df['is_anomaly'] == 0, 'score'].to_numpy()\n    h_threshold_z = float(np.percentile(_h_val_normal_z, float(config['scoring']['threshold_quantile']) * 100))\n    h_labels = h_test_df['is_anomaly'].to_numpy()\n    h_scores = h_test_df['score'].to_numpy()\n    h_metrics = compute_threshold_metrics(h_labels, h_scores, h_threshold_z)\n    h_breakdown = build_defect_breakdown(holdout_dataset['test_defect_types'], h_labels, h_scores, h_threshold_z)\n    h_sweep_df = sweep_percentile_thresholds(h_val_df.loc[h_val_df['is_anomaly'] == 0, 'score'].to_numpy(), h_labels, h_scores)\n    _h_best_sweep = h_sweep_df.sort_values('f1', ascending=false).iloc[0].to_dict()\n    h_eval_dir.mkdir(parents=true, exist_ok=true)\n    h_plots_dir.mkdir(parents=true, exist_ok=true)\n    h_val_df.to_csv(h_eval_dir / 'val_scores.csv', index=false)\n    h_test_df.to_csv(h_eval_dir / 'test_scores.csv', index=false)\n    h_breakdown.to_csv(h_eval_dir / 'saved_defect_breakdown.csv', index=false)\n    h_sweep_df.to_csv(h_eval_dir / 'threshold_sweep.csv', index=false)\n    h_summary = {'evaluation_mode': 'holdout70k_3p5k', 'threshold_z': float(h_threshold_z), 'threshold_raw': float(h_threshold_z * _h_z_std + _h_z_mean), 'precision': float(h_metrics['precision']), 'recall': float(h_metrics['recall']), 'f1': float(h_metrics['f1']), 'roc_auc_z': float(h_metrics['roc_auc']), 'avg_precision_z': float(h_metrics['avg_precision']), 'n_test_normal': int((h_labels == 0).sum()), 'n_test_defect': int((h_labels == 1).sum()), 'threshold_sweep_best': _h_best_sweep}\n    (holdout_dir / 'results').mkdir(parents=true, exist_ok=true)\n    (holdout_dir / 'results' / 'summary.json').write_text(json.dumps(h_summary, indent=2))\n    print(f'holdout artifacts saved to {holdout_dir}')\nelif _h_csvs_exist:\n    h_summary = json.loads(_h_summary_path.read_text())\n    h_val_df = pd.read_csv(h_eval_dir / 'val_scores.csv')\n    h_test_df = pd.read_csv(h_eval_dir / 'test_scores.csv')\n    h_sweep_df = pd.read_csv(h_eval_dir / 'threshold_sweep.csv')\n    h_breakdown = pd.read_csv(h_eval_dir / 'saved_defect_breakdown.csv')\n    h_threshold_z = float(h_summary['threshold_z'])\n    h_labels = h_test_df['is_anomaly'].to_numpy()\n    h_scores = h_test_df['score'].to_numpy()\n    h_metrics = compute_threshold_metrics(h_labels, h_scores, h_threshold_z)\n    _h_best = float(h_summary.get('threshold_sweep_best', {}).get('threshold_z', h_threshold_z))\n    if not run_holdout:\n        print('run_holdout=false but pre-saved holdout artifacts found - loading and plotting.')\nelse:\n    print('holdout skipped: run_holdout=false and no pre-saved artifacts found.')\n    h_summary = none\nif 'h_summary' in dir() and h_summary is not none:\n    h_plots_dir.mkdir(parents="
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## UMAP Diagnostics

- `REGENERATE_UMAP = False` (default): display pre-saved UMAP PNGs from `plots/`
- `REGENERATE_UMAP = True`: replot from saved UMAP coordinate CSV at `results/umap/umap_test_embeddings.csv` (no GPU needed)


In [ ]:
try:
    from IPython.display import Image as IPImage
    _split_png = PLOTS_DIR / 'umap_test_embeddings.png'
    _score_png = PLOTS_DIR / 'umap_by_score.png'
    _umap_csv = UMAP_DIR / 'umap_test_embeddings.csv'
    _need_replot = REGENERATE_UMAP or (not _split_png.exists() and _umap_csv.exists())
    if not _need_replot:
        for _png in [_split_png, _score_png]:
            if _png.exists():
                print(f'Displaying: {_png.name}')
                display(IPImage(filename=str(_png)))
            else:
                print(f'Not found: {_png}. Set REGENERATE_UMAP=True or ensure the UMAP CSV exists at:')
                print(f'  {_umap_csv}')
    elif not _umap_csv.exists():
        print(f'UMAP CSV not found: {_umap_csv}')
        print('UMAP CSV and plots are not available in the saved artifacts.')
    else:
        _df = pd.read_csv(_umap_csv)
        _x_col = 'umap_1' if 'umap_1' in _df.columns else 'umap1'
        _y_col = 'umap_2' if 'umap_2' in _df.columns else 'umap2'
        _x, _y = (_df[_x_col].values, _df[_y_col].values)
        _split_col = next((c for c in ['split_label', 'group'] if c in _df.columns), None)
        if _split_col is not None:
            _is_anomaly = _df[_split_col].isin(['test_anomaly']).values
        elif 'is_anomaly' in _df.columns:
            _is_anomaly = _df['is_anomaly'].values.astype(bool)
        else:
            _is_anomaly = np.zeros(len(_df), dtype=bool)
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(_x[~_is_anomaly], _y[~_is_anomaly], s=3, alpha=0.15, c='#aaaaaa', linewidths=0, label='Normal')
        ax.scatter(_x[_is_anomaly], _y[_is_anomaly], s=20, alpha=0.85, c='#e41a1c', linewidths=0, label='Anomaly')
        ax.set_title('ViT-B/16 PatchCore UMAP -- Split-coloured (cosine metric)', fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.tight_layout()
        fig.savefig(_split_png, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f'Saved: {_split_png}')
        _score_col = next((c for c in ['score', 'model_score'] if c in _df.columns), None)
        if _score_col is not None:
            _has_score = _df[_score_col].notna().values
            fig, ax = plt.subplots(figsize=(7, 6))
            sc = ax.scatter(_x[_has_score], _y[_has_score], c=_df[_score_col].values[_has_score].astype(float), cmap='hot_r', s=3, alpha=0.6, linewidths=0)
            plt.colorbar(sc, ax=ax, label='Anomaly score (z-normalised)')
            ax.set_title('ViT-B/16 PatchCore UMAP -- Score-coloured (cosine metric)', fontweight='bold')
            ax.set_xticks([])
            ax.set_yticks([])
            plt.tight_layout()
            fig.savefig(_score_png, dpi=150, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            print(f'Saved: {_score_png}')
        else:
            print('No score column in UMAP CSV -- score-coloured plot skipped.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from ipython.display import image as ipimage\n_split_png = plots_dir / 'umap_test_embeddings.png'\n_score_png = plots_dir / 'umap_by_score.png'\n_umap_csv = umap_dir / 'umap_test_embeddings.csv'\n_need_replot = regenerate_umap or (not _split_png.exists() and _umap_csv.exists())\nif not _need_replot:\n    for _png in [_split_png, _score_png]:\n        if _png.exists():\n            print(f'displaying: {_png.name}')\n            display(ipimage(filename=str(_png)))\n        else:\n            print(f'not found: {_png}. set regenerate_umap=true or ensure the umap csv exists at:')\n            print(f'  {_umap_csv}')\nelif not _umap_csv.exists():\n    print(f'umap csv not found: {_umap_csv}')\n    print('umap csv and plots are not available in the saved artifacts.')\nelse:\n    _df = pd.read_csv(_umap_csv)\n    _x_col = 'umap_1' if 'umap_1' in _df.columns else 'umap1'\n    _y_col = 'umap_2' if 'umap_2' in _df.columns else 'umap2'\n    _x, _y = (_df[_x_col].values, _df[_y_col].values)\n    _split_col = next((c for c in ['split_label', 'group'] if c in _df.columns), none)\n    if _split_col is not none:\n        _is_anomaly = _df[_split_col].isin(['test_anomaly']).values\n    elif 'is_anomaly' in _df.columns:\n        _is_anomaly = _df['is_anomaly'].values.astype(bool)\n    else:\n        _is_anomaly = np.zeros(len(_df), dtype=bool)\n    fig, ax = plt.subplots(figsize=(7, 6))\n    ax.scatter(_x[~_is_anomaly], _y[~_is_anomaly], s=3, alpha=0.15, c='#aaaaaa', linewidths=0, label='normal')\n    ax.scatter(_x[_is_anomaly], _y[_is_anomaly], s=20, alpha=0.85, c='#e41a1c', linewidths=0, label='anomaly')\n    ax.set_title('vit-b/16 patchcore umap -- split-coloured (cosine metric)', fontweight='bold')\n    ax.legend(fontsize=9)\n    ax.set_xticks([])\n    ax.set_yticks([])\n    plt.tight_layout()\n    fig.savefig(_split_png, dpi=150, bbox_inches='tight')\n    plt.show()\n    plt.close(fig)\n    print(f'saved: {_split_png}')\n    _score_col = next((c for c in ['score', 'model_score'] if c in _df.columns), none)\n    if _score_col is not none:\n        _has_score = _df[_score_col].notna().values\n        fig, ax = plt.subplots(figsize=(7, 6))\n        sc = ax.scatter(_x[_has_score], _y[_has_score], c=_df[_score_col].values[_has_score].astype(float), cmap='hot_r', s=3, alpha=0.6, linewidths=0)\n        plt.colorbar(sc, ax=ax, label='anomaly score (z-normalised)')\n        ax.set_title('vit-b/16 patchcore umap -- score-coloured (cosine metric)', fontweight='bold')\n        ax.set_xticks([])\n        ax.set_yticks([])\n        plt.tight_layout()\n        fig.savefig(_score_png, dpi=150, bbox_inches='tight')\n        plt.show()\n        plt.close(fig)\n        print(f'saved: {_score_png}')\n    else:\n        print('no score column in umap csv -- score-coloured plot skipped.')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Save Outputs


In [ ]:
try:
    import shutil
    STANDARD_ARTIFACT_DIR = REPO_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x224/main/artifacts'
    STANDARD_CHECKPOINTS_DIR = STANDARD_ARTIFACT_DIR / 'checkpoints'
    STANDARD_PLOTS_DIR = STANDARD_ARTIFACT_DIR / 'plots'
    STANDARD_RESULTS_DIR = STANDARD_ARTIFACT_DIR / 'results'
    for _path in [STANDARD_CHECKPOINTS_DIR, STANDARD_PLOTS_DIR, STANDARD_RESULTS_DIR]:
        _path.mkdir(parents=True, exist_ok=True)
    source_model_path = CHKPT_DIR / 'model.pt'
    if not source_model_path.exists():
        source_model_path = CHKPT_DIR / 'best_model.pt'
    target_model_path = STANDARD_CHECKPOINTS_DIR / 'model.pt'
    if source_model_path.exists():
        _artifact = torch.load(source_model_path, map_location='cpu')
        torch.save(_artifact, target_model_path)
    if 'test_scores_df' in globals():
        test_scores_df.to_csv(STANDARD_RESULTS_DIR / 'test_scores.csv', index=False)
    if 'breakdown' in globals():
        breakdown.to_csv(STANDARD_RESULTS_DIR / 'test_defect_recall.csv', index=False)
    summary_payload = dict(summary)
    summary_payload.update({'model_export_path': str(target_model_path), 'test_scores_csv_path': str(STANDARD_RESULTS_DIR / 'test_scores.csv'), 'defect_recall_csv_path': str(STANDARD_RESULTS_DIR / 'test_defect_recall.csv')})
    (STANDARD_RESULTS_DIR / 'evaluation_metrics.json').write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')
    source_umap_csv = UMAP_DIR / 'umap_test_embeddings.csv'
    source_umap_png = PLOTS_DIR / 'umap_test_embeddings.png'
    if source_umap_csv.exists():
        shutil.copyfile(source_umap_csv, STANDARD_RESULTS_DIR / 'umap_test_embeddings.csv')
    if source_umap_png.exists():
        shutil.copyfile(source_umap_png, STANDARD_PLOTS_DIR / 'umap_test_embeddings.png')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import shutil\nstandard_artifact_dir = repo_root / 'experiments/anomaly_detection/patchcore/vit_b16/x224/main/artifacts'\nstandard_checkpoints_dir = standard_artifact_dir / 'checkpoints'\nstandard_plots_dir = standard_artifact_dir / 'plots'\nstandard_results_dir = standard_artifact_dir / 'results'\nfor _path in [standard_checkpoints_dir, standard_plots_dir, standard_results_dir]:\n    _path.mkdir(parents=true, exist_ok=true)\nsource_model_path = chkpt_dir / 'model.pt'\nif not source_model_path.exists():\n    source_model_path = chkpt_dir / 'best_model.pt'\ntarget_model_path = standard_checkpoints_dir / 'model.pt'\nif source_model_path.exists():\n    _artifact = torch.load(source_model_path, map_location='cpu')\n    torch.save(_artifact, target_model_path)\nif 'test_scores_df' in globals():\n    test_scores_df.to_csv(standard_results_dir / 'test_scores.csv', index=false)\nif 'breakdown' in globals():\n    breakdown.to_csv(standard_results_dir / 'test_defect_recall.csv', index=false)\nsummary_payload = dict(summary)\nsummary_payload.update({'model_export_path': str(target_model_path), 'test_scores_csv_path': str(standard_results_dir / 'test_scores.csv'), 'defect_recall_csv_path': str(standard_results_dir / 'test_defect_recall.csv')})\n(standard_results_dir / 'evaluation_metrics.json').write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')\nsource_umap_csv = umap_dir / 'umap_test_embeddings.csv'\nsource_umap_png = plots_dir / 'umap_test_embeddings.png'\nif source_umap_csv.exists():\n    shutil.copyfile(source_umap_csv, standard_results_dir / 'umap_test_embeddings.csv')\nif source_umap_png.exists():\n    shutil.copyfile(source_umap_png, standard_plots_dir / 'umap_test_embeddings.png')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
